In [12]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [15]:
words = open('names.txt', 'r').read().splitlines()
print(words[:5])
print(len(words))

['emma', 'olivia', 'ava', 'isabella', 'sophia']
32033


In [34]:
# Choose n-gram context window
n = 4

# Build vocab of itos and stoi
chars = sorted(list(set(''.join(words))))

# Vocabulary and mappings between string and integer
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}


# Dataset of X -> Y mappings
X, Y = [], []

"""
For each name, keep a sliding window context of the last n characters.
For the first n-1 characters, populate the empty places with '.'
E.g. for n = 4, the sliding context windows for "cedric" would be
. . . c
. . c e
. c e d
c e d r
e d r i
d r i c
""";

for word in words[:5]:

    print(word)
    context = [0] * n
    for char in word:
        ix = stoi[char]
        X.append (context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
.... ---> e
...e ---> m
..em ---> m
.emm ---> a
olivia
.... ---> o
...o ---> l
..ol ---> i
.oli ---> v
oliv ---> i
livi ---> a
ava
.... ---> a
...a ---> v
..av ---> a
isabella
.... ---> i
...i ---> s
..is ---> a
.isa ---> b
isab ---> e
sabe ---> l
abel ---> l
bell ---> a
sophia
.... ---> s
...s ---> o
..so ---> p
.sop ---> h
soph ---> i
ophi ---> a


In [ ]:
"""
Now we have the dataset we need to start training. Lets build our model.

We're building our model following Bengio et al. 2003 (A Neural Probabilistic Language Model)
where instead of words, we will have a vocab of 27 characters (including the special char '.').

We will choose a context window n for each word.
Each input x is a list of n characters that maps to an ouptut y which is the (n+1)-th character in the word.

Each character is one-hot encoded and multiplied by a 27 x 10 lookup table C to be embedded in a 10 imensional vector space.

We will pass our inputs of chosen context window through a MLP of 4 layers of 100 neurons.
Each layer will have a linear transformation wx + b, and non-linear activation function tanh.
With deeper layers of linear transformation, we will have increasing variance. 
This will lead to a greater saturation of the tails of tanh, which will kill the neuron.
We need to combat this by normalizing the distribution of activations before passing them to tanh.

There are a few ways to do this:

Kaiming initialization, whereby we construct the standard deviation of the sampling distribution that 
weights are drawn from initialization by multiplying by "empirical gain" and dividing it by the square root of fan-in (number of inputs)

The "empirical gain" counteracts for the lost variance due to the squashing of the nonlinearity
For our tanh, this constant happens to be 5/3 (due to some math that figured out how much tanh squashes).
For a nonllinearity like ReLU (which squashes all negatives), it would be sqrt(2) to compensate for all the lost negatives.

Since output variance = input variance * fan-in, our target weight variance is 1/fan-in to cancel the linear inflation due to fan-in.
To get our target weight standard deviation, we square root by the target weight variance, which is sqrt(1/fan-in) = 1/sqrt(fan-in).

So we multiply by 5/3 which is the empirical gain to add back the variance lost due to tanh and divide by
sqrt(fan-in) to counter variance inflation due to fan-in to give us our Kaiming initialization for this model.
Therefore, the standard deviation of the sampling distribution that we draw our weights from should be k_std = (5/3)/sqrt(fan-in)

This solves the weight variance problem at initialization. However, as traiing progresses, weights will change and alter the distribution.
We need a dynamic method to correct this shifting distribution. Following Ioffe et al. 2015 (Batch Normalization...), we can use the
batch normalization method to dynamically correct the shifting distribution at every step. 

For each batch of inputs, we can center the mean at 0 and normalize the standard deviation of the activations.
Each activation x_i is normalized by subtracting the mean and dividing by the square root of the variance
plus a small constant epsilon to prevent division by zero if the variance is zero: x_i_hat = (x_i - mean)/sqrt(variance + epsilon)

But this creates another problem. The mean and standard deviation is batch dependent.
During training, each input is evaluated in the context of its own batch for its mean and standard deviation.

However, during inference, whereby we evaluate a single input at a time, we do not have local context from other inputs.
If we just threw the input in and let it pass by the activation layers unnormalized, it would result in garbage since 
the model's parameters were tuned on normalized activations.

So we keep track of a global "running mean" and "running variance" that we update during training
which we can use to normalize single input inferences that work with the parameters of the model.

The purpose of normalizing the inputs is to let it start off with a "baseline" normal input to the nonlinearity
so it doesn't get stuck at the "tails" of the activation function. Over time, as the network learns,
it will update some scaling paramter gamma and shifting paramter beta, slowly shifting it to the nonlinear tails of the nonlinearity.
If we didn't normalize it first, the variance could instantly throw it at tails of the nonlinearity and never return. 
""";

